# MTCA-2 · Stage 7b · Semantic Frame-Stability (re-analysis)

**Purpose:** re-measure frame stability using **embedding cosine similarity** (meaning) instead of **Jaccard** (vocabulary), to test the Stage 8 council's qualitative claim that frame-variation is *surface over a stable reasoning core*.

**Method: mirrors lexical Stage 7 exactly** — same 5-field `extract_response_text`, same per-(specimen,model) mean-pairwise-across-8-frames structure. The **only** change is the similarity measure: cosine over `all-MiniLM-L6-v2` sentence embeddings (normalized) in place of Jaccard.

**Prediction (H8-semantic):** if framing produces surface variation over a stable core, semantic stability should be **substantially higher** than lexical stability.

**No new API calls.** Deterministic re-analysis over the 520 committed Stage 6 responses.

**Reproducibility caveat:** embedding computation is floating-point and not bit-identical across hardware/library versions. Values are frozen at 4-decimal precision and reproduce to that precision; the embedding model and version are recorded in the artifact. This is a weaker reproducibility guarantee than the lexical Stage 7 (pure set operations), and is stated as such.


## S7b-0 — Load committed responses

In [ ]:
# S7b-0
!pip -q install sentence-transformers scipy

import os, re, json, hashlib, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
REPO = Path("/content/mtca-research") if IN_COLAB else Path("./mtca-research")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/billyrdavis1985-bot/mtca-research.git",str(REPO)], check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"], check=True)

RESP = REPO/"mtca-2"/"stage6_execution"/"responses"
resp = {}
for f in os.listdir(RESP):
    r = json.load(open(RESP/f))
    resp[(r['specimen_id'], r['frame_id'], r['model_id'])] = r
units  = sorted(set(k[0] for k in resp))
frames = sorted(set(k[1] for k in resp))
models = sorted(set(k[2] for k in resp))
assert len(resp) == 520
print(f"Loaded {len(resp)} responses: {len(units)} specimens x {len(frames)} frames x {len(models)} models")

## S7b-1 — Text extraction (identical to lexical Stage 7) + lexical baseline

In [ ]:
# S7b-1
def extract_response_text(response):
    parsed = response.get('parsed_json', {})
    if not parsed: return ''
    parts = []
    for field in ['communicative_function','coherence_assessment']:
        v = parsed.get(field,'')
        if isinstance(v,str): parts.append(v)
    for field in ['implicit_claims','conditions_for_usefulness','conditions_for_misleading']:
        v = parsed.get(field,[])
        if isinstance(v,list): parts.extend(str(x) for x in v)
        elif isinstance(v,str): parts.append(v)
    return ' '.join(parts)

# lexical Jaccard (for parallel comparison — identical to Stage 7)
def tokenize(t): return set(w for w in re.findall(r'\b[a-z]{3,}\b', t.lower()))
def jaccard(a,b):
    ta,tb = tokenize(a),tokenize(b)
    if not ta and not tb: return 1.0
    if not ta or not tb:  return 0.0
    return len(ta&tb)/len(ta|tb)
print("extraction + lexical baseline ready")

## S7b-2 — Embed all responses (all-MiniLM-L6-v2, normalized)

In [ ]:
# S7b-2
from sentence_transformers import SentenceTransformer
EMBED_MODEL = 'all-MiniLM-L6-v2'
model = SentenceTransformer(EMBED_MODEL)

keys  = list(resp.keys())
texts = [extract_response_text(resp[k]) for k in keys]
emb   = model.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
emb_lookup = {keys[i]: emb[i] for i in range(len(keys))}
def cosine(a,b): return float(np.dot(a,b))   # normalized -> dot = cosine
print(f"Embedded {len(texts)} responses with {EMBED_MODEL}")

## S7b-3 — Frame stability: lexical vs semantic

In [ ]:
# S7b-3
lex_stab, sem_stab = {}, {}
for u in units:
    for m in models:
        fr = [f for f in frames if (u,f,m) in resp]
        if len(fr) < 2: continue
        lt = [extract_response_text(resp[(u,f,m)]) for f in fr]
        lex_stab[(u,m)] = float(np.mean([jaccard(lt[i],lt[j]) for i in range(len(lt)) for j in range(i+1,len(lt))]))
        ev = [emb_lookup[(u,f,m)] for f in fr]
        sem_stab[(u,m)] = float(np.mean([cosine(ev[i],ev[j]) for i in range(len(ev)) for j in range(i+1,len(ev))]))

lex_mean = float(np.mean(list(lex_stab.values())))
sem_mean = float(np.mean(list(sem_stab.values())))
print("=== HEADLINE ===")
print(f"Lexical  (Jaccard): {lex_mean:.4f}")
print(f"Semantic (cosine):  {sem_mean:.4f}")
print(f"Delta:              {sem_mean-lex_mean:+.4f}   Ratio: {sem_mean/lex_mean:.2f}x")

print("\n=== Per-model (lexical -> semantic) ===")
for m in models:
    lv = float(np.mean([v for (u,mm),v in lex_stab.items() if mm==m]))
    sv = float(np.mean([v for (u,mm),v in sem_stab.items() if mm==m]))
    print(f"  {m:20} {lv:.4f} -> {sv:.4f}")

from scipy.stats import spearmanr, pearsonr
pairs = sorted(lex_stab.keys())
lv = np.array([lex_stab[p] for p in pairs]); sv = np.array([sem_stab[p] for p in pairs])
rho,pval = spearmanr(lv,sv); r,_ = pearsonr(lv,sv)
print(f"\nMetric agreement (n={len(pairs)}): Spearman rho={rho:.3f} (p={pval:.2e}), Pearson r={r:.3f}")

## S7b-4 — Freeze artifact

In [ ]:
# S7b-4
STAGE7B = REPO/"mtca-2"/"stage7b_semantic"; STAGE7B.mkdir(parents=True, exist_ok=True)
artifact = {
    'analysis':'stage7b_semantic_similarity','study':'MTCA-2',
    'embedding_model':EMBED_MODEL,
    'method':'mean pairwise cosine similarity across 8 frame responses per (specimen,model); mirrors lexical Stage 7, similarity measure swapped',
    'reproducibility_note':'floating-point; values frozen at 4-decimal precision; not bit-identical across environments (weaker than lexical Stage 7)',
    'lexical_mean_stability': round(lex_mean,4),
    'semantic_mean_stability': round(sem_mean,4),
    'semantic_minus_lexical': round(sem_mean-lex_mean,4),
    'ratio_semantic_over_lexical': round(sem_mean/lex_mean,2),
    'spearman_rho': round(float(rho),3), 'pearson_r': round(float(r),3),
    'per_model': {m:{'lexical':round(float(np.mean([v for (u,mm),v in lex_stab.items() if mm==m])),4),
                     'semantic':round(float(np.mean([v for (u,mm),v in sem_stab.items() if mm==m])),4)} for m in models},
    'per_specimen_semantic': {u:round(float(np.mean([v for (uu,m),v in sem_stab.items() if uu==u])),4) for u in units},
    'creation_date': datetime.now(timezone.utc).isoformat(),
    'language_discipline_note':'Model behavior under specified methodology. No claims about framework validity.',
}
ser = json.dumps({k:v for k,v in artifact.items() if k!='creation_date'}, sort_keys=True, ensure_ascii=False)
sha = hashlib.sha256(ser.encode()).hexdigest(); artifact['artifact_sha256'] = sha
(STAGE7B/f"stage7b_semantic_{sha[:16]}.json").write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding='utf-8')
print(f"Frozen: stage7b_semantic_{sha[:16]}.json")
print(f"SHA256: {sha}")
print("\nInterpretation: semantic stability >> lexical stability confirms the Stage 8 council's")
print("qualitative 'surface variation over stable core' finding with a 4th independent method.")